## Analysis 1 
as of Jan 26, user follower distribution? 
user profile, incremental from last snapshot date? 
buyer segment, new users, pro users, 
sellers segment, 

In [0]:
spark.conf.set("spark.sql.shuffle.partitions","1024")
spark.conf.set("spark.sql.sources.partitionOverwriteMode","dynamic")

In [0]:
# COMMAND ----------
# Parameters

output_base = 's3://data-tmp-poshmark-production/qiong/social/follow/'
snapshot_dt = '2026-01-01'

# Optional cleanup helper
def materialize_df(df, s3_path, view_name, mode="overwrite", partition_col="snapshot_dt"):
    (
        df.repartition(partition_col)
          .write
          .mode(mode)
          .partitionBy(partition_col)
          .format("parquet")
          .save(s3_path)
    )
    queryDF = spark.read.parquet(s3_path)
    queryDF.createOrReplaceTempView(view_name)
    return queryDF

In [0]:
# COMMAND ----------
# 1) Base follow snapshot

BaseFollowDF = spark.sql(f"""
SELECT
    DATE('{snapshot_dt}') AS snapshot_dt,
    user_id        AS followee_user_id,
    follower_id    AS follower_user_id,
    initiated_by,
    latest_follow_timestamp,
    latest_unfollow_timestamp,
    TO_DATE(latest_follow_timestamp) AS follow_dt,
    TO_DATE(NULLIF(latest_unfollow_timestamp, '-')) AS unfollow_dt,
    CASE WHEN NULLIF(latest_unfollow_timestamp, '-') IS NULL THEN 1 ELSE 0 END AS is_current_follow,
    CASE WHEN initiated_by = 'system' THEN 1 ELSE 0 END AS is_system_initiated,
    CASE WHEN initiated_by = 'user' THEN 1 ELSE 0 END AS is_user_initiated
FROM s3_analytics_snapshots.dw_user_follow_history_quarterly
WHERE snapshot_date = DATE('{snapshot_dt}')
""")

s3_path = output_base + 'base_follow/'
materialize_df(BaseFollowDF, s3_path, 'base_follow')

In [0]:
# COMMAND ----------
# 2) Distinct user universe

DistinctUsersDF = spark.sql(f"""
SELECT DISTINCT user_id
FROM (
    SELECT follower_user_id AS user_id FROM base_follow
    UNION ALL
    SELECT followee_user_id AS user_id FROM base_follow
) u
""")

s3_path = output_base + 'distinct_users/'
(
    DistinctUsersDF.write
    .mode("overwrite")
    .format("parquet")
    .save(s3_path)
)
spark.read.parquet(s3_path).createOrReplaceTempView("distinct_users")

In [0]:
# COMMAND ----------
# 3) User lifetime attributes

UserAttrDF = spark.sql(f"""
SELECT
    dw.user_id,
    dw.generation,
    CASE
        WHEN dw.gender IN ('male', 'male (hidden)') THEN 'male'
        ELSE 'female'
    END AS user_gender,
    TO_DATE(COALESCE(dw.guest_joined_at, dw.joined_at)) AS joined_date,
    TO_DATE(dw.buyer_activated_at)  AS buyer_activated_date,
    TO_DATE(dw.lister_activated_at) AS lister_activated_date,
    TO_DATE(dw.seller_activated_at) AS seller_activated_date,
    dw.acq_channel_group_v2_lc,
    dw.acq_channel_group_lc,
    dw.acq_channel_lc
FROM s3_analytics.dw_users dw
INNER JOIN distinct_users u
    ON dw.user_id = u.user_id
WHERE dw.home_domain = 'us'
  AND dw.is_valid_user IS TRUE
  AND NOT COALESCE(
        DATEDIFF(
            CASE WHEN dw.user_status = 'restricted' THEN dw.status_updated_at ELSE NULL END,
            COALESCE(dw.guest_joined_at, dw.joined_at)
        ) + 1 <= 30,
        FALSE
  )
""")

s3_path = output_base + 'user_attr/'
UserAttrDF.write.mode("overwrite").format("parquet").save(s3_path)
spark.read.parquet(s3_path).createOrReplaceTempView("user_attr")

In [0]:
# COMMAND ----------
# 4) Buyer segment lookup

BuyerSegDF = spark.sql(f"""
SELECT
    buyer_id,
    shopper_segment_daily
FROM analytics_scratch.l365d_shopper_segment
WHERE DATE('{snapshot_dt}') > TO_DATE(start_date)
  AND DATE('{snapshot_dt}') <= TO_DATE(COALESCE(end_date, CURRENT_DATE()))
""")

s3_path = output_base + 'buyer_segment/'
BuyerSegDF.write.mode("overwrite").format("parquet").save(s3_path)
spark.read.parquet(s3_path).createOrReplaceTempView("buyer_segment")

In [0]:
# COMMAND ----------
# 5) Seller segment lookup

SellerSegDF = spark.sql(f"""
SELECT
    user_id,
    user_segment_daily AS seller_segment_daily
FROM s3_analytics.dw_user_segments
WHERE user_type = 'seller'
  AND segment_date = DATE('{snapshot_dt}')
""")

s3_path = output_base + 'seller_segment/'
SellerSegDF.write.mode("overwrite").format("parquet").save(s3_path)
spark.read.parquet(s3_path).createOrReplaceTempView("seller_segment")

In [0]:
# COMMAND ----------
# 6) AU segment lookup

AuSegDF = spark.sql(f"""
SELECT
    user_id,
    au_segment AS daily_au_segment
FROM external_scratch.core__users_active_user_segments_daily
WHERE segment_date = DATE('{snapshot_dt}')
""")

s3_path = output_base + 'au_segment/'
AuSegDF.write.mode("overwrite").format("parquet").save(s3_path)
spark.read.parquet(s3_path).createOrReplaceTempView("au_segment")

In [0]:
# COMMAND ----------
# 7) Current network sizes

FollowerCountsDF = spark.sql("""
SELECT
    followee_user_id,
    COUNT(*) AS follower_cnt
FROM base_follow
WHERE is_current_follow = 1
GROUP BY 1
""")

s3_path = output_base + 'follower_counts/'
FollowerCountsDF.write.mode("overwrite").format("parquet").save(s3_path)
spark.read.parquet(s3_path).createOrReplaceTempView("follower_counts")

FollowingCountsDF = spark.sql("""
SELECT
    follower_user_id,
    COUNT(*) AS following_cnt
FROM base_follow
WHERE is_current_follow = 1
GROUP BY 1
""")

s3_path = output_base + 'following_counts/'
FollowingCountsDF.write.mode("overwrite").format("parquet").save(s3_path)
spark.read.parquet(s3_path).createOrReplaceTempView("following_counts")

In [0]:
# COMMAND ----------
# 8) Final enriched follow table

FollowEnrichedDF = spark.sql("""
SELECT
    f.snapshot_dt,
    f.followee_user_id,
    f.follower_user_id,
    f.initiated_by,
    f.latest_follow_timestamp,
    f.latest_unfollow_timestamp,
    f.follow_dt,
    f.unfollow_dt,
    f.is_current_follow,
    f.is_system_initiated,
    f.is_user_initiated,

    -- follower attributes
    fa.generation                  AS follower_generation,
    fa.user_gender                 AS follower_gender,
    fa.joined_date                 AS follower_joined_date,
    fa.buyer_activated_date        AS follower_buyer_activated_date,
    fa.lister_activated_date       AS follower_lister_activated_date,
    fa.seller_activated_date       AS follower_seller_activated_date,
    fa.acq_channel_group_v2_lc     AS follower_acq_channel_group_v2_lc,
    fa.acq_channel_group_lc        AS follower_acq_channel_group_lc,
    fa.acq_channel_lc              AS follower_acq_channel_lc,
    bs.shopper_segment_daily       AS follower_buyer_segment_daily,
    au.daily_au_segment            AS follower_au_segment_daily,

    -- followee attributes
    ea.generation                  AS followee_generation,
    ea.user_gender                 AS followee_gender,
    ea.joined_date                 AS followee_joined_date,
    ea.buyer_activated_date        AS followee_buyer_activated_date,
    ea.lister_activated_date       AS followee_lister_activated_date,
    ea.seller_activated_date       AS followee_seller_activated_date,
    ea.acq_channel_group_v2_lc     AS followee_acq_channel_group_v2_lc,
    ea.acq_channel_group_lc        AS followee_acq_channel_group_lc,
    ea.acq_channel_lc              AS followee_acq_channel_lc,
    ss.seller_segment_daily        AS followee_seller_segment_daily,

    -- size features
    COALESCE(fc.follower_cnt, 0)   AS follower_cnt,
    COALESCE(fg.following_cnt, 0)  AS following_cnt,

    -- derived features
    DATEDIFF(f.snapshot_dt, f.follow_dt) AS follow_age_days,

    CASE
        WHEN DATEDIFF(f.snapshot_dt, f.follow_dt) <= 7   THEN 'a. 0-7d'
        WHEN DATEDIFF(f.snapshot_dt, f.follow_dt) <= 30  THEN 'b. 8-30d'
        WHEN DATEDIFF(f.snapshot_dt, f.follow_dt) <= 90  THEN 'c. 31-90d'
        WHEN DATEDIFF(f.snapshot_dt, f.follow_dt) <= 365 THEN 'd. 91-365d'
        ELSE 'e. 365d+'
    END AS follow_age_bucket,

    CASE
        WHEN COALESCE(fg.following_cnt, 0) <= 10    THEN 'a. 1-10'
        WHEN COALESCE(fg.following_cnt, 0) <= 100   THEN 'b. 11-100'
        WHEN COALESCE(fg.following_cnt, 0) <= 1000  THEN 'c. 101-1k'
        WHEN COALESCE(fg.following_cnt, 0) <= 10000 THEN 'd. 1k-10k'
        ELSE 'e. >10k'
    END AS follower_size_bucket,

    CASE
        WHEN COALESCE(fc.follower_cnt, 0) <= 10    THEN 'a. 1-10'
        WHEN COALESCE(fc.follower_cnt, 0) <= 100   THEN 'b. 11-100'
        WHEN COALESCE(fc.follower_cnt, 0) <= 1000  THEN 'c. 101-1k'
        WHEN COALESCE(fc.follower_cnt, 0) <= 10000 THEN 'd. 1k-10k'
        ELSE 'e. >10k'
    END AS followee_size_bucket,

    CASE
        WHEN fa.joined_date IS NULL THEN 'unknown'
        WHEN DATEDIFF(f.snapshot_dt, fa.joined_date) <= 30  THEN 'new'
        WHEN DATEDIFF(f.snapshot_dt, fa.joined_date) <= 180 THEN 'mid'
        ELSE 'tenured'
    END AS follower_tenure_bucket,

    CASE
        WHEN ea.joined_date IS NULL THEN 'unknown'
        WHEN DATEDIFF(f.snapshot_dt, ea.joined_date) <= 30  THEN 'new'
        WHEN DATEDIFF(f.snapshot_dt, ea.joined_date) <= 180 THEN 'mid'
        ELSE 'tenured'
    END AS followee_tenure_bucket

FROM base_follow f
LEFT JOIN user_attr fa
    ON f.follower_user_id = fa.user_id
LEFT JOIN user_attr ea
    ON f.followee_user_id = ea.user_id
LEFT JOIN buyer_segment bs
    ON f.follower_user_id = bs.buyer_id
LEFT JOIN au_segment au
    ON f.follower_user_id = au.user_id
LEFT JOIN seller_segment ss
    ON f.followee_user_id = ss.user_id
LEFT JOIN follower_counts fc
    ON f.followee_user_id = fc.followee_user_id
LEFT JOIN following_counts fg
    ON f.follower_user_id = fg.follower_user_id
""")

s3_path = output_base + 'follow_enriched/'
FollowEnrichedDF.write.mode("overwrite").format("parquet").save(s3_path)
spark.read.parquet(s3_path).createOrReplaceTempView("follow_enriched")

In [0]:
# COMMAND ----------
# 9) Distribution summary

FollowDistributionDF = spark.sql("""
SELECT
    initiated_by,
    follower_gender,
    follower_generation,
    follower_buyer_segment_daily,
    follower_au_segment_daily,
    followee_gender,
    followee_generation,
    followee_seller_segment_daily,
    follower_tenure_bucket,
    followee_tenure_bucket,
    follower_size_bucket,
    followee_size_bucket,
    follow_age_bucket,
    COUNT(*) AS follow_events,
    COUNT(DISTINCT follower_user_id) AS distinct_followers,
    COUNT(DISTINCT followee_user_id) AS distinct_followees,
    AVG(CASE WHEN is_current_follow = 1 THEN 1.0 ELSE 0.0 END) AS current_follow_rate
FROM follow_enriched
GROUP BY
    initiated_by,
    follower_gender,
    follower_generation,
    follower_buyer_segment_daily,
    follower_au_segment_daily,
    followee_gender,
    followee_generation,
    followee_seller_segment_daily,
    follower_tenure_bucket,
    followee_tenure_bucket,
    follower_size_bucket,
    followee_size_bucket,
    follow_age_bucket
""")

s3_path = output_base + 'follow_distribution/'
FollowDistributionDF.write.mode("overwrite").format("parquet").save(s3_path)
spark.read.parquet(s3_path).createOrReplaceTempView("follow_distribution")

In [0]:
# COMMAND ----------
# 10) Source split

FollowSourceSplitDF = spark.sql("""
SELECT
    initiated_by,
    COUNT(*) AS follow_events,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_of_total
FROM follow_enriched
GROUP BY 1
ORDER BY follow_events DESC
""")

s3_path = output_base + 'follow_source_split/'
FollowSourceSplitDF.write.mode("overwrite").format("parquet").save(s3_path)
spark.read.parquet(s3_path).createOrReplaceTempView("follow_source_split")

In [0]:
# COMMAND ----------
# 11) Size distribution

FollowSizeDistributionDF = spark.sql("""
SELECT
    follower_size_bucket,
    followee_size_bucket,
    COUNT(*) AS follow_events
FROM follow_enriched
GROUP BY 1, 2
ORDER BY follow_events DESC
""")

s3_path = output_base + 'follow_size_distribution/'
FollowSizeDistributionDF.write.mode("overwrite").format("parquet").save(s3_path)
spark.read.parquet(s3_path).createOrReplaceTempView("follow_size_distribution")

In [0]:
# COMMAND ----------
# 12) Quick checks

display(FollowEnrichedDF.limit(100))
display(FollowDistributionDF.limit(100))
display(FollowSourceSplitDF)
display(FollowSizeDistributionDF)

In [0]:
%sql
select *
from s3_analytics_snapshots.dw_user_follow_history_quarterly
where snapshot_date = '2026-01-01'
limit 10;

In [0]:
%sql 
select snapshot_date, count(*) 
from s3_analytics_snapshots.dw_user_follow_history_quarterly
group by 1
order by 1 desc 
limit 5

In [0]:
-- =========================================================
-- Follow analysis using snapshot_date = '2026-01-01'
-- Base table schema:
--   user_id, follower_id, initiated_by,
--   latest_follow_timestamp, latest_unfollow_timestamp, snapshot_date
-- =========================================================


queryDF  = spark.sql(f"""WITH base_follow AS (
    SELECT
        DATE '2026-01-01' AS snapshot_dt,

        /* In your schema:
           user_id     = followee / followed account
           follower_id = follower / actor who follows
        */
        user_id        AS followee_user_id,
        follower_id    AS follower_user_id,
        initiated_by,

        TRY_TO_TIMESTAMP_TZ(latest_follow_timestamp)   AS latest_follow_ts,
        TRY_TO_TIMESTAMP_TZ(NULLIF(latest_unfollow_timestamp, '-')) AS latest_unfollow_ts,

        CAST(TRY_TO_TIMESTAMP_TZ(latest_follow_timestamp) AS DATE) AS follow_dt,
        CAST(TRY_TO_TIMESTAMP_TZ(NULLIF(latest_unfollow_timestamp, '-')) AS DATE) AS unfollow_dt,

        CASE
            WHEN TRY_TO_TIMESTAMP_TZ(NULLIF(latest_unfollow_timestamp, '-')) IS NULL THEN 1
            ELSE 0
        END AS is_current_follow,

        CASE
            WHEN initiated_by = 'system' THEN 1
            ELSE 0
        END AS is_system_initiated,

        CASE
            WHEN initiated_by = 'user' THEN 1
            ELSE 0
        END AS is_user_initiated

    FROM s3_analytics_snapshots.dw_user_follow_history_quarterly
    WHERE snapshot_date = DATE '2026-01-01'
),

user_attr AS (
    SELECT
        user_id,
        generation,
        CASE
            WHEN gender IN ('male', 'male (hidden)') THEN 'male'
            ELSE 'female'
        END AS user_gender,
        cohort_date,
        DATE_TRUNC('day', lister_activated_at::date)::date AS lister_activated_dt,
        DATE_TRUNC('day', buyer_activated_at::date)::date  AS buyer_activated_dt,
        acq_channel_group_v2_lc,
        acq_channel_group_lc,
        acq_channel_lc
    FROM s3_analytics.dw_users
    WHERE home_domain = 'us'
      AND is_valid_user IS TRUE
      AND NOT COALESCE(
            DATEDIFF(
                (CASE WHEN user_status = 'restricted' THEN status_updated_at ELSE NULL END),
                COALESCE(guest_joined_at, joined_at)
            ) + 1 <= 30,
            FALSE
      )
),

buyer_segment AS (
    SELECT
        buyer_id,
        shopper_segment_daily
    FROM analytics_scratch.l365d_shopper_segment
    WHERE DATE '2026-01-01' > DATE(start_date)
      AND DATE '2026-01-01' <= DATE(COALESCE(end_date, CURRENT_DATE()))
),

daily_au_segment AS (
    SELECT
        user_id,
        au_segment AS daily_au_segment
    FROM external_scratch.core__users_active_user_segments_daily
    WHERE segment_date = DATE '2026-01-01'
),

seller_segment AS (
    SELECT
        user_id,
        user_segment_daily AS seller_segment_daily
    FROM s3_analytics.dw_user_segments
    WHERE user_type = 'seller'
      AND segment_date = DATE '2026-01-01'
)

SELECT
    f.snapshot_dt,
    f.followee_user_id,
    f.follower_user_id,
    f.initiated_by,
    f.latest_follow_ts,
    f.latest_unfollow_ts,
    f.follow_dt,
    f.unfollow_dt,
    f.is_current_follow,
    f.is_system_initiated,
    f.is_user_initiated,

    /* follower attributes */
    fa.generation              AS follower_generation,
    fa.user_gender             AS follower_gender,
    fa.cohort_date             AS follower_cohort_date,
    fa.lister_activated_dt      AS follower_lister_activated_dt,
    fa.buyer_activated_dt      AS follower_buyer_activated_dt,
    fa.acq_channel_group_v2_lc  AS follower_acq_channel_group_v2_lc,
    fa.acq_channel_group_lc     AS follower_acq_channel_group_lc,
    fa.acq_channel_lc           AS follower_acq_channel_lc,
    bs.shopper_segment_daily    AS follower_buyer_segment_daily,
    au.daily_au_segment         AS follower_au_segment_daily,

    /* followee attributes */
    ea.generation               AS followee_generation,
    ea.user_gender              AS followee_gender,
    ea.cohort_date              AS followee_cohort_date,
    ea.lister_activated_dt      AS followee_lister_activated_dt,
    ea.buyer_activated_dt       AS followee_buyer_activated_dt,
    ea.acq_channel_group_v2_lc  AS followee_acq_channel_group_v2_lc,
    ea.acq_channel_group_lc     AS followee_acq_channel_group_lc,
    ea.acq_channel_lc          AS followee_acq_channel_lc,
    ss.seller_segment_daily     AS followee_seller_segment_daily

FROM base_follow f
LEFT JOIN user_attr fa
    ON f.follower_user_id = fa.user_id
LEFT JOIN user_attr ea
    ON f.followee_user_id = ea.user_id
LEFT JOIN buyer_segment bs
    ON f.follower_user_id = bs.buyer_id
LEFT JOIN daily_au_segment au
    ON f.follower_user_id = au.user_id
LEFT JOIN seller_segment ss
    ON f.followee_user_id = ss.user_id
""")


In [0]:
%sql
CREATE OR REPLACE TEMP TABLE follow_snapshot_enriched AS

In [0]:

-- =========================================================
-- 1) Basic distribution
-- =========================================================

SELECT
    COUNT(*) AS follow_events,
    COUNT(DISTINCT follower_user_id) AS distinct_followers,
    COUNT(DISTINCT followee_user_id)  AS distinct_followees,
    AVG(CASE WHEN is_current_follow = 1 THEN 1 ELSE 0 END) AS current_follow_rate,
    AVG(CASE WHEN is_system_initiated = 1 THEN 1 ELSE 0 END) AS system_initiated_rate,
    AVG(CASE WHEN is_user_initiated = 1 THEN 1 ELSE 0 END) AS user_initiated_rate
FROM follow_snapshot_enriched
;

-- =========================================================
-- 2) Distribution by initiated_by
-- =========================================================

SELECT
    initiated_by,
    COUNT(*) AS follow_events,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_of_all_follows,
    COUNT(DISTINCT follower_user_id) AS distinct_followers,
    COUNT(DISTINCT followee_user_id)  AS distinct_followees
FROM follow_snapshot_enriched
GROUP BY 1
ORDER BY follow_events DESC
;

-- =========================================================
-- 3) Distribution by follower / followee attributes
-- =========================================================

SELECT
    initiated_by,
    follower_gender,
    follower_generation,
    follower_buyer_segment_daily,
    follower_au_segment_daily,
    followee_gender,
    followee_generation,
    followee_seller_segment_daily,

    COUNT(*) AS follow_events,
    COUNT(DISTINCT follower_user_id) AS distinct_followers,
    COUNT(DISTINCT followee_user_id)  AS distinct_followees,

    AVG(CASE WHEN is_current_follow = 1 THEN 1 ELSE 0 END) AS current_follow_rate
FROM follow_snapshot_enriched
GROUP BY
    initiated_by,
    follower_gender,
    follower_generation,
    follower_buyer_segment_daily,
    follower_au_segment_daily,
    followee_gender,
    followee_generation,
    followee_seller_segment_daily
ORDER BY follow_events DESC
')
